# Pizza Orders with Kafka and Avro

This notebook demonstrates a complete, deliberately small event-streaming flow:

1. verify Kafka, Confluent Schema Registry, and Kafka Connect;
2. recreate one `pizza-orders` topic;
3. publish five pizza orders using an Avro schema;
4. confirm the schema registered under `pizza-orders-value`; and
5. let a kitchen consumer and an analytics consumer independently process every order.

> Run `docker compose up -d` from the repository root and wait for all services to become healthy before running these cells.

In [1]:
import json
import time
from collections import Counter
from typing import Any
from urllib.request import urlopen

from confluent_kafka import Consumer, KafkaError, KafkaException, Producer
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroDeserializer, AvroSerializer
from confluent_kafka.serialization import MessageField, SerializationContext

BOOTSTRAP_SERVERS = 'localhost:9092'
SCHEMA_REGISTRY_URL = 'http://localhost:8081'
CONNECT_URL = 'http://localhost:8083'
TOPIC = 'pizza-orders'
EXPECTED_ORDER_COUNT = 5


def get_json(url: str) -> Any:
    try:
        with urlopen(url, timeout = 5) as response:
            return json.load(response)
    except OSError as exc:
        raise RuntimeError(f'Could not reach {url}. Is the Docker Compose stack healthy?') from exc

## 1. Verify the local services

Kafka clients connect through the host listener. Schema Registry and Kafka Connect expose small HTTP APIs that make useful readiness checks. Connect is running and configured, but this first demo does not create a connector.

In [2]:
admin = AdminClient({'bootstrap.servers': BOOTSTRAP_SERVERS})
kafka_metadata = admin.list_topics(timeout = 10)
schema_registry_subjects = get_json(f'{SCHEMA_REGISTRY_URL}/subjects')
connect_info = get_json(CONNECT_URL)
connect_plugins = get_json(f'{CONNECT_URL}/connector-plugins')

assert kafka_metadata.brokers, 'Kafka returned no broker metadata.'
assert isinstance(schema_registry_subjects, list)
assert isinstance(connect_plugins, list)

{
    'kafka_brokers': sorted(kafka_metadata.brokers),
    'schema_registry_subjects_before_run': schema_registry_subjects,
    'kafka_connect_version': connect_info['version'],
    'kafka_connect_plugins': [plugin['class'] for plugin in connect_plugins],
}

{'kafka_brokers': [1],
 'schema_registry_subjects_before_run': ['pizza-orders-value'],
 'kafka_connect_version': '8.3.1-ccs',
 'kafka_connect_plugins': ['org.apache.kafka.connect.mirror.MirrorCheckpointConnector',
  'org.apache.kafka.connect.mirror.MirrorHeartbeatConnector',
  'org.apache.kafka.connect.mirror.MirrorSourceConnector']}

## 2. Recreate the demo topic

Deleting and recreating only `pizza-orders` makes repeated notebook runs deterministic. The topic has one partition, so its five records retain producer order, and a replication factor of one to match the single broker.

In [3]:
def topic_exists(admin_client: AdminClient, topic: str) -> bool:
    metadata = admin_client.list_topics(timeout = 5)
    topic_metadata = metadata.topics.get(topic)
    return topic_metadata is not None and topic_metadata.error is None


def reset_topic(admin_client: AdminClient, topic: str) -> None:
    if topic_exists(admin_client, topic):
        admin_client.delete_topics([topic], operation_timeout = 10)[topic].result()

        delete_deadline = time.monotonic() + 15
        while topic_exists(admin_client, topic) and time.monotonic() < delete_deadline:
            time.sleep(0.5)
        if topic_exists(admin_client, topic):
            raise TimeoutError(f'Topic {topic!r} was not deleted within 15 seconds.')

    new_topic = NewTopic(topic, num_partitions = 1, replication_factor = 1)
    admin_client.create_topics([new_topic], operation_timeout = 10)[topic].result()


reset_topic(admin, TOPIC)
topic_metadata = admin.list_topics(topic = TOPIC, timeout = 10).topics[TOPIC]
assert topic_metadata.error is None
assert len(topic_metadata.partitions) == 1
f'Created {TOPIC!r} with {len(topic_metadata.partitions)} partition.'

"Created 'pizza-orders' with 1 partition."

## 3. Define the PizzaOrder contract

The producer passes this Avro schema to Confluent's serializer. On the first message, the serializer registers it with Schema Registry using the default topic-name strategy. Money is represented as integer cents to avoid floating-point rounding.

In [4]:
PIZZA_ORDER_SCHEMA = {
    'type': 'record',
    'namespace': 'com.example.pizza',
    'name': 'PizzaOrder',
    'fields': [
        {'name': 'order_id', 'type': 'string'},
        {'name': 'customer_name', 'type': 'string'},
        {'name': 'pizza_name', 'type': 'string'},
        {'name': 'size', 'type': 'string'},
        {'name': 'quantity', 'type': 'int'},
        {'name': 'toppings', 'type': {'type': 'array', 'items': 'string'}},
        {
            'name': 'special_instructions',
            'type': ['null', 'string'],
            'default': None,
        },
        {'name': 'fulfillment_method', 'type': 'string'},
        {'name': 'unit_price_cents', 'type': 'int'},
        {'name': 'created_at', 'type': 'string'},
    ],
}

sample_orders = [
    {
        'order_id': 'order-1001',
        'customer_name': 'Alex',
        'pizza_name': 'Margherita',
        'size': 'medium',
        'quantity': 1,
        'toppings': ['fresh basil'],
        'special_instructions': None,
        'fulfillment_method': 'pickup',
        'unit_price_cents': 1400,
        'created_at': '2026-08-29T18:00:00Z',
    },
    {
        'order_id': 'order-1002',
        'customer_name': 'Blair',
        'pizza_name': 'Pepperoni',
        'size': 'large',
        'quantity': 2,
        'toppings': ['extra cheese'],
        'special_instructions': 'Bake one pizza well done.',
        'fulfillment_method': 'delivery',
        'unit_price_cents': 1850,
        'created_at': '2026-08-29T18:01:00Z',
    },
    {
        'order_id': 'order-1003',
        'customer_name': 'Casey',
        'pizza_name': 'Veggie Supreme',
        'size': 'small',
        'quantity': 1,
        'toppings': ['mushrooms', 'bell peppers', 'olives'],
        'special_instructions': 'No onions.',
        'fulfillment_method': 'pickup',
        'unit_price_cents': 1500,
        'created_at': '2026-08-29T18:02:00Z',
    },
    {
        'order_id': 'order-1004',
        'customer_name': 'Devon',
        'pizza_name': 'Hawaiian',
        'size': 'large',
        'quantity': 1,
        'toppings': ['jalapeños'],
        'special_instructions': None,
        'fulfillment_method': 'delivery',
        'unit_price_cents': 1750,
        'created_at': '2026-08-29T18:03:00Z',
    },
    {
        'order_id': 'order-1005',
        'customer_name': 'Emery',
        'pizza_name': 'Pepperoni',
        'size': 'medium',
        'quantity': 3,
        'toppings': ['mushrooms'],
        'special_instructions': 'Cut into squares.',
        'fulfillment_method': 'pickup',
        'unit_price_cents': 1600,
        'created_at': '2026-08-29T18:04:00Z',
    },
]

assert len(sample_orders) == EXPECTED_ORDER_COUNT
PIZZA_ORDER_SCHEMA

{'type': 'record',
 'namespace': 'com.example.pizza',
 'name': 'PizzaOrder',
 'fields': [{'name': 'order_id', 'type': 'string'},
  {'name': 'customer_name', 'type': 'string'},
  {'name': 'pizza_name', 'type': 'string'},
  {'name': 'size', 'type': 'string'},
  {'name': 'quantity', 'type': 'int'},
  {'name': 'toppings', 'type': {'type': 'array', 'items': 'string'}},
  {'name': 'special_instructions',
   'type': ['null', 'string'],
   'default': None},
  {'name': 'fulfillment_method', 'type': 'string'},
  {'name': 'unit_price_cents', 'type': 'int'},
  {'name': 'created_at', 'type': 'string'}]}

## 4. Produce five Avro messages

Each Kafka key is the order ID. The delivery callback captures the broker-assigned partition and offset, while `flush` ensures every queued record is acknowledged before continuing.

In [5]:
schema_registry_client = SchemaRegistryClient({'url': SCHEMA_REGISTRY_URL})
avro_serializer = AvroSerializer(
    schema_registry_client,
    json.dumps(PIZZA_ORDER_SCHEMA),
)
producer = Producer({'bootstrap.servers': BOOTSTRAP_SERVERS})
delivery_results: list[dict[str, Any]] = []


def record_delivery(error: KafkaError | None, message: Any) -> None:
    delivery_results.append(
        {
            'key': message.key().decode('utf-8'),
            'topic': message.topic(),
            'partition': message.partition(),
            'offset': message.offset(),
            'error': str(error) if error is not None else None,
        }
    )


serialization_context = SerializationContext(TOPIC, MessageField.VALUE)
for order in sample_orders:
    producer.produce(
        topic = TOPIC,
        key = order['order_id'].encode('utf-8'),
        value = avro_serializer(order, serialization_context),
        on_delivery = record_delivery,
    )
    producer.poll(0)

undelivered_count = producer.flush(timeout = 10)
failed_deliveries = [result for result in delivery_results if result['error'] is not None]

if undelivered_count:
    raise TimeoutError(f'{undelivered_count} Kafka messages were not delivered within 10 seconds.')
if failed_deliveries:
    raise RuntimeError(f'Kafka rejected messages: {failed_deliveries}')
assert len(delivery_results) == EXPECTED_ORDER_COUNT

delivery_results.sort(key = lambda result: result['offset'])
delivery_results

[{'key': 'order-1001',
  'topic': 'pizza-orders',
  'partition': 0,
  'offset': 0,
  'error': None},
 {'key': 'order-1002',
  'topic': 'pizza-orders',
  'partition': 0,
  'offset': 1,
  'error': None},
 {'key': 'order-1003',
  'topic': 'pizza-orders',
  'partition': 0,
  'offset': 2,
  'error': None},
 {'key': 'order-1004',
  'topic': 'pizza-orders',
  'partition': 0,
  'offset': 3,
  'error': None},
 {'key': 'order-1005',
  'topic': 'pizza-orders',
  'partition': 0,
  'offset': 4,
  'error': None}]

## 5. Inspect the registered schema

Confluent's wire format stores a schema ID with each message. Consumers use that ID to retrieve the matching writer schema, so producer and consumer code do not have to exchange schema files directly.

In [6]:
expected_subject = f'{TOPIC}-value'
registered_subjects = get_json(f'{SCHEMA_REGISTRY_URL}/subjects')
latest_schema = get_json(
    f'{SCHEMA_REGISTRY_URL}/subjects/{expected_subject}/versions/latest'
)
registered_schema = json.loads(latest_schema['schema'])

assert expected_subject in registered_subjects
assert registered_schema == PIZZA_ORDER_SCHEMA

{
    'subject': latest_schema['subject'],
    'version': latest_schema['version'],
    'schema_id': latest_schema['id'],
    'record_name': registered_schema['name'],
    'field_names': [field['name'] for field in registered_schema['fields']],
}

{'subject': 'pizza-orders-value',
 'version': 1,
 'schema_id': 1,
 'record_name': 'PizzaOrder',
 'field_names': ['order_id',
  'customer_name',
  'pizza_name',
  'size',
  'quantity',
  'toppings',
  'special_instructions',
  'fulfillment_method',
  'unit_price_cents',
  'created_at']}

## 6. Consume the orders independently

Both consumers subscribe to the same topic but use different group IDs. Kafka therefore delivers the full five-order stream to each group. Polling has a fixed deadline, automatic offset commits are disabled, and `close()` always runs.

In [7]:
avro_deserializer = AvroDeserializer(schema_registry_client)


def consume_orders(
    group_id: str,
    expected_count: int,
    timeout_seconds: float = 20.0,
) -> list[dict[str, Any]]:
    consumer = Consumer(
        {
            'bootstrap.servers': BOOTSTRAP_SERVERS,
            'group.id': group_id,
            'auto.offset.reset': 'earliest',
            'enable.auto.commit': False,
        }
    )
    consumed_orders: list[dict[str, Any]] = []
    deadline = time.monotonic() + timeout_seconds

    try:
        consumer.subscribe([TOPIC])
        while len(consumed_orders) < expected_count and time.monotonic() < deadline:
            message = consumer.poll(timeout = 1.0)
            if message is None:
                continue
            if message.error():
                if message.error().code() == KafkaError._PARTITION_EOF:
                    continue
                raise KafkaException(message.error())

            order = avro_deserializer(
                message.value(),
                SerializationContext(message.topic(), MessageField.VALUE),
            )
            message_key = message.key().decode('utf-8')
            if message_key != order['order_id']:
                raise ValueError(f'Key {message_key!r} does not match the order ID.')

            order['_kafka_partition'] = message.partition()
            order['_kafka_offset'] = message.offset()
            consumed_orders.append(order)
    finally:
        consumer.close()

    if len(consumed_orders) != expected_count:
        raise TimeoutError(
            f'{group_id!r} consumed {len(consumed_orders)} of {expected_count} '
            f'orders within {timeout_seconds:.0f} seconds.'
        )
    return consumed_orders

### Consumer A: kitchen fulfillment

The kitchen turns each event into a compact preparation ticket.

In [8]:
kitchen_orders = consume_orders('pizza-kitchen', EXPECTED_ORDER_COUNT)
assert [order['order_id'] for order in kitchen_orders] == [
    order['order_id'] for order in sample_orders
]

for order in kitchen_orders:
    toppings = ', '.join(order['toppings']) or 'standard toppings'
    instructions = order['special_instructions'] or 'None'
    print(
        f"{order['order_id']} | {order['quantity']}x {order['size']} "
        f"{order['pizza_name']} | {toppings} | {order['fulfillment_method'].upper()} "
        f"| Notes: {instructions}"
    )

order-1001 | 1x medium Margherita | fresh basil | PICKUP | Notes: None
order-1002 | 2x large Pepperoni | extra cheese | DELIVERY | Notes: Bake one pizza well done.
order-1003 | 1x small Veggie Supreme | mushrooms, bell peppers, olives | PICKUP | Notes: No onions.
order-1004 | 1x large Hawaiian | jalapeños | DELIVERY | Notes: None
order-1005 | 3x medium Pepperoni | mushrooms | PICKUP | Notes: Cut into squares.


### Consumer B: sales analytics

A separate group receives the same records and converts them into a small operational summary.

In [9]:
analytics_orders = consume_orders('pizza-analytics', EXPECTED_ORDER_COUNT)
pizza_quantities: Counter[str] = Counter()
fulfillment_orders: Counter[str] = Counter()

for order in analytics_orders:
    pizza_quantities[order['pizza_name']] += order['quantity']
    fulfillment_orders[order['fulfillment_method']] += 1

revenue_cents = sum(
    order['quantity'] * order['unit_price_cents'] for order in analytics_orders
)
analytics_summary = {
    'orders_processed': len(analytics_orders),
    'pizzas_ordered': sum(pizza_quantities.values()),
    'pizza_quantities': dict(pizza_quantities),
    'gross_revenue': f'${revenue_cents / 100:,.2f}',
    'fulfillment_orders': dict(fulfillment_orders),
}

assert analytics_summary == {
    'orders_processed': 5,
    'pizzas_ordered': 8,
    'pizza_quantities': {
        'Margherita': 1,
        'Pepperoni': 5,
        'Veggie Supreme': 1,
        'Hawaiian': 1,
    },
    'gross_revenue': '$131.50',
    'fulfillment_orders': {'pickup': 3, 'delivery': 2},
}

analytics_summary

{'orders_processed': 5,
 'pizzas_ordered': 8,
 'pizza_quantities': {'Margherita': 1,
  'Pepperoni': 5,
  'Veggie Supreme': 1,
  'Hawaiian': 1},
 'gross_revenue': '$131.50',
 'fulfillment_orders': {'pickup': 3, 'delivery': 2}}

## Demo complete

Five Avro records were acknowledged by Kafka, their schema was verified in Schema Registry, and two independent consumer groups each processed the complete stream.